# 03 — Exploratory data analysis

**Role: Descriptive and spatial-temporal EDA.**

This notebook turns the validated national-panel EDA into a readable data-science review. It reads real project artefacts and makes small in-memory summaries and figures; it does **not** rebuild the national panel or select a model.

The analytical record is one 1 km mainland cell × predictor year. Fire recurrence is measured in a mainland-masked 2 km context.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.notebook_support import read_json_artifact, resolve_project_root
PROJECT_ROOT = resolve_project_root(PROJECT_ROOT)
eda = read_json_artifact(PROJECT_ROOT, 'reports/validation/national_panel_model_readiness_eda.json')
print(f"Loaded validated EDA for {eda['row_count']:,} cell-year rows.")

## Panel and missing-data contract

These are persisted validation results, not hand-entered numbers. Missing predictor values would block model readiness.

In [ ]:
split_rows = pd.Series(eda['split_row_counts'], name='cell_year_rows').to_frame()
missingness = pd.Series(eda['missingness'], name='missing_values').to_frame()
display(split_rows)
display(missingness)
assert missingness['missing_values'].eq(0).all(), 'Validated panel has unexpected predictor missingness.'
print('Model-design gate:', eda['model_design_decision']['gate'])

## Target distribution through time

`burned_share_next_year` is continuous and zero-heavy. The chart is descriptive only; it does not fit or tune a model.

In [ ]:
target_by_year = pd.DataFrame.from_dict(eda['target']['by_year'], orient='index').sort_index()
target_by_year.index.name = 'predictor_year'
display(target_by_year)

fig, ax = plt.subplots(figsize=(9, 4))
target_by_year['zero_proportion'].plot(kind='bar', ax=ax, color='#a65628')
ax.set(title='Zero share of next-year burned-area target by predictor year', xlabel='Predictor year T', ylabel='Zero-target proportion', ylim=(0, 1))
plt.show()
print('Overall zero proportion:', f"{eda['target']['overall_zero_proportion']:.1%}")

## Predictor relationships

Correlation is a screening diagnostic, not causal evidence. It is useful for identifying obvious redundancy before modelling.

In [ ]:
correlations = pd.DataFrame(eda['correlations'])
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(correlations, vmin=-1, vmax=1, cmap='coolwarm')
ax.set_xticks(range(len(correlations.columns)), correlations.columns, rotation=90)
ax.set_yticks(range(len(correlations.index)), correlations.index)
fig.colorbar(image, ax=ax, label='Pearson correlation')
ax.set_title('Validated national-panel predictor correlations')
fig.tight_layout()
plt.show()
print('Pairs with |correlation| ≥ 0.8:', eda['high_redundancy_pairs_abs_ge_0_8'] or 'none recorded')

## Reproducibility boundary

To deliberately rebuild the panel and EDA from local raw inputs, use `python scripts/run_project.py --mode reproduce --confirm-rebuild` in a terminal. Notebook execution is intentionally read-and-explain by default so it cannot accidentally trigger a national geospatial rebuild.